In [ ]:
from __future__ import annotations

import ast
import sqlite3
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd

DB_PATH = Path('moltbook_27_30.db')
OUTPUT_DIR = Path('.')
START_DATE = '2026-01-27'
CUTOFF_DATE = '2026-03-21'
EXCEL_OUTPUT = OUTPUT_DIR / 'RobustezzaNoIsolati.xlsx'
REMOVAL_PERCENTS = (0.00, 0.01, 0.05, 0.10, 0.20, 0.40, 0.60, 0.80)
ROBUSTNESS_RANDOM_RUNS = 3

In [ ]:
def connect_sqlite_read_only(db_path: Path) -> sqlite3.Connection:
    if not db_path.exists():
        raise FileNotFoundError(f'Database not found: {db_path}')
    uri = f'file:{db_path.as_posix()}?mode=ro'
    conn = sqlite3.connect(uri, uri=True)
    conn.execute('PRAGMA query_only = ON')
    conn.execute('PRAGMA foreign_keys = ON')
    return conn

def not_deleted_sql(alias: str) -> str:
    return (
        f'({alias}.is_deleted IS NULL OR '
        f"(LOWER(CAST({alias}.is_deleted AS TEXT)) NOT IN ('1', 'true')) )"
    )

def safe_replies(value: object) -> list[dict]:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    if isinstance(value, str):
        try:
            value = ast.literal_eval(value.strip())
        except (SyntaxError, ValueError):
            return []
    if isinstance(value, dict):
        return [value]
    if isinstance(value, list):
        return [item for item in value if isinstance(item, dict)]
    return []

def collect_replies(payload: object, post_id: object, post_author_id: object, rows: list[dict]) -> None:
    for reply in safe_replies(payload):
        author = reply.get('author') if isinstance(reply.get('author'), dict) else {}
        author_id = reply.get('author_id') or author.get('id')
        created_at = reply.get('created_at')
        rows.append({
            'day': created_at,
            'source': author_id,
            'target': post_author_id,
            'is_deleted': reply.get('is_deleted'),
        })
        collect_replies(reply.get('replies'), post_id, post_author_id, rows)

def fetch_recursive_reply_interactions(conn: sqlite3.Connection, start_date: str, cutoff_date: str) -> pd.DataFrame:
    query = f'''
        SELECT c.replies, CAST(p.author_id AS TEXT) AS post_author_id
        FROM comments_new AS c
        INNER JOIN posts AS p ON c.post_id = p.id
        WHERE DATE(c.created_at) >= ? AND DATE(c.created_at) <= ?
          AND {not_deleted_sql('c')} AND {not_deleted_sql('p')}
          AND p.author_id IS NOT NULL AND c.replies IS NOT NULL
    '''
    base = pd.read_sql_query(query, conn, params=(start_date, cutoff_date))
    rows: list[dict] = []
    for item in base.itertuples(index=False):
        collect_replies(item.replies, None, item.post_author_id, rows)
    if not rows:
        return pd.DataFrame(columns=['day', 'source', 'target'])
    replies = pd.DataFrame(rows)
    replies['day'] = pd.to_datetime(replies['day'], errors='coerce', utc=True).dt.tz_convert(None).dt.normalize()
    replies['source'] = replies['source'].astype('string').str.strip()
    replies['target'] = replies['target'].astype('string').str.strip()
    replies = replies[~replies['is_deleted'].map(lambda value: str(value).strip().lower() in {'1', 'true'})]
    return replies.dropna(subset=['day', 'source', 'target']).query('source != target')[['day', 'source', 'target']]

def load_interactions_from_db(db_path: Path, start_date: str, cutoff_date: str) -> pd.DataFrame:
    with connect_sqlite_read_only(db_path) as conn:
        query = f'''
            SELECT DATE(c.created_at) AS day, CAST(c.author_id AS TEXT) AS source,
                   CAST(p.author_id AS TEXT) AS target
            FROM comments_new AS c
            INNER JOIN posts AS p ON c.post_id = p.id
            WHERE DATE(c.created_at) >= ? AND DATE(c.created_at) <= ?
              AND {not_deleted_sql('c')} AND {not_deleted_sql('p')}
              AND c.author_id IS NOT NULL AND p.author_id IS NOT NULL
        '''
        interactions = pd.read_sql_query(query, conn, params=(start_date, cutoff_date))
        replies = fetch_recursive_reply_interactions(conn, start_date, cutoff_date)
    interactions = pd.concat([interactions, replies], ignore_index=True)
    interactions['day'] = pd.to_datetime(interactions['day'], errors='coerce', utc=True).dt.tz_convert(None).dt.normalize()
    interactions['source'] = interactions['source'].astype('string').str.strip()
    interactions['target'] = interactions['target'].astype('string').str.strip()
    return interactions.dropna(subset=['day', 'source', 'target']).query('source != target')

def load_interactions(output_dir: Path, db_path: Path, start_date: str, cutoff_date: str) -> pd.DataFrame:
    parquet_path = output_dir / 'interactions1.parquet'
    if parquet_path.exists():
        interactions = pd.read_parquet(parquet_path)
        interactions['day'] = pd.to_datetime(interactions['day'], errors='coerce', utc=True).dt.tz_convert(None).dt.normalize()
        interactions['source'] = interactions['source'].astype('string').str.strip()
        interactions['target'] = interactions['target'].astype('string').str.strip()
    else:
        interactions = load_interactions_from_db(db_path, start_date, cutoff_date)
    start_day = pd.Timestamp(start_date).normalize()
    cutoff_day = pd.Timestamp(cutoff_date).normalize()
    return interactions[(interactions['day'] >= start_day) & (interactions['day'] <= cutoff_day)].dropna(subset=['source', 'target']).query('source != target').copy()

In [ ]:
def largest_cc_fraction(graph: nx.Graph) -> float:
    node_count = graph.number_of_nodes()
    if node_count == 0:
        return 0.0
    return max((len(component) for component in nx.connected_components(graph)), default=0) / node_count

def robustness_snapshot(graph_u: nx.Graph, graph_d: nx.DiGraph, removal_percents=REMOVAL_PERCENTS, random_runs=ROBUSTNESS_RANDOM_RUNS, seed=42) -> pd.DataFrame:
    # The input graph already contains only non-isolated nodes.
    active_nodes = [node for node, degree in graph_u.degree() if degree > 0]
    graph_u = graph_u.subgraph(active_nodes).copy()
    graph_d = graph_d.subgraph(active_nodes).copy()
    n0 = graph_u.number_of_nodes()
    columns = ['attack', 'percent_removed', 'removed_nodes', 'gc_fraction', 'gc_fraction_mean', 'gc_fraction_std']
    if n0 == 0:
        return pd.DataFrame(columns=columns)
    rng = np.random.default_rng(seed)
    nodes = np.array(list(graph_u.nodes()))
    indegree_ranked = [node for node, _ in sorted(graph_d.in_degree(), key=lambda item: item[1], reverse=True)]
    outdegree_ranked = [node for node, _ in sorted(graph_d.out_degree(), key=lambda item: item[1], reverse=True)]
    rows = []
    for percent in removal_percents:
        removed_count = min(n0, max(1, int(round(percent * n0))))
        random_values = []
        for _ in range(random_runs):
            removed = set(rng.choice(nodes, size=removed_count, replace=False))
            reduced = graph_u.copy()
            reduced.remove_nodes_from(removed)
            random_values.append(largest_cc_fraction(reduced))
        random_mean = float(np.mean(random_values))
        rows.append({'attack': 'random', 'percent_removed': percent, 'removed_nodes': removed_count, 'gc_fraction': random_mean, 'gc_fraction_mean': random_mean, 'gc_fraction_std': float(np.std(random_values))})
        for attack, ranked_nodes in [('targeted_indegree', indegree_ranked), ('targeted_outdegree', outdegree_ranked)]:
            reduced = graph_u.copy()
            reduced.remove_nodes_from(set(ranked_nodes[:removed_count]))
            value = largest_cc_fraction(reduced)
            rows.append({'attack': attack, 'percent_removed': percent, 'removed_nodes': removed_count, 'gc_fraction': value, 'gc_fraction_mean': value, 'gc_fraction_std': 0.0})
    return pd.DataFrame(rows, columns=columns)

def calculate_robustness_daily(interactions: pd.DataFrame, start_date: str, cutoff_date: str) -> pd.DataFrame:
    interactions = interactions.sort_values(['day', 'source', 'target']).reset_index(drop=True)
    all_days = pd.date_range(pd.Timestamp(start_date), pd.Timestamp(cutoff_date), freq='D')
    graph_u = nx.Graph()
    graph_d = nx.DiGraph()
    rows = []
    for day in all_days:
        day_rows = interactions[interactions['day'] == day]
        for source, target in day_rows[['source', 'target']].itertuples(index=False):
            graph_d.add_edge(source, target)
            graph_u.add_edge(source, target)
        active_nodes = [node for node, degree in graph_u.degree() if degree > 0]
        if active_nodes:
            snapshot = robustness_snapshot(graph_u, graph_d)
            if not snapshot.empty:
                snapshot.insert(0, 'day', day)
                rows.append(snapshot)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

In [4]:
interactions = load_interactions(OUTPUT_DIR, DB_PATH, START_DATE, CUTOFF_DATE)
robustness_daily = calculate_robustness_daily(interactions, START_DATE, CUTOFF_DATE)

with pd.ExcelWriter(EXCEL_OUTPUT, engine='openpyxl') as writer:
    robustness_daily.to_excel(writer, sheet_name='robustness_daily', index=False)

display(robustness_daily.head())

,day,attack,percent_removed,removed_nodes,gc_fraction,gc_fraction_mean,gc_fraction_std
0,2026-01-28,random,0.00,1,0.982456,0.982456,0.024811
1,2026-01-28,targeted_indegree,0.00,1,0.947368,0.947368,0.000000
2,2026-01-28,targeted_outdegree,0.00,1,0.947368,0.947368,0.000000
3,2026-01-28,random,0.01,1,0.964912,0.964912,0.024811
4,2026-01-28,targeted_indegree,0.01,1,0.947368,0.947368,0.000000
